In [2]:
### Data ingestion to Vector DB pipeline

In [3]:
import os
from importlib.metadata import metadata

from langchain_classic.agents.agent_toolkits.vectorstore import prompt
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_core import documents
from langchain_core.vectorstores import VectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/home/om/PycharmProjects/RAG-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
###Read all pdfs in the directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    ### Find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    ### Final count to see all pdf files
    print(f"Found {len(pdf_files)} PDF files")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            ### Adding more information related to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nAll documents loaded: {len(all_documents)}")
    return all_documents

### Process all pdfs in data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files
Processing ../data/pdf/mlops_roadmap.pdf
Loaded 21 pages

All documents loaded: 21


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'GPL Ghostscript 9.56.1', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': "D:20251016072905Z00'00'", 'moddate': "D:20251016072905Z00'00'", 'source': '../data/pdf/mlops_roadmap.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'mlops_roadmap.pdf', 'file_type': 'pdf'}, page_content='MLOps Roadmap\nYour Beginner-Friendly Guide to Machine Learning Operations\nby TechWorld with Nana'),
 Document(metadata={'producer': 'GPL Ghostscript 9.56.1', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': "D:20251016072905Z00'00'", 'moddate': "D:20251016072905Z00'00'", 'source': '../data/pdf/mlops_roadmap.pdf', 'total_pages': 21, 'page': 1, 'page_label': '2', 'source_file': 'mlops_roadmap.pdf', 'file_type': 'pdf'}, page_content="Provided by TechWorld with Nana\nI'm Nana, Co-Founder of TechWorld with Nana.\nAs a Cloud and DevOps engineer, I'm dedicated to \nhelping engineers build the most valuable and 

In [6]:
### Text splitting into chunks
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Splitting documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [7]:
chunks = split_documents(all_pdf_documents)
chunks

Split 21 documents into 36 chunks

Example chunk:
Content: MLOps Roadmap
Your Beginner-Friendly Guide to Machine Learning Operations
by TechWorld with Nana...
Metadata: {'producer': 'GPL Ghostscript 9.56.1', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': "D:20251016072905Z00'00'", 'moddate': "D:20251016072905Z00'00'", 'source': '../data/pdf/mlops_roadmap.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'mlops_roadmap.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'GPL Ghostscript 9.56.1', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': "D:20251016072905Z00'00'", 'moddate': "D:20251016072905Z00'00'", 'source': '../data/pdf/mlops_roadmap.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'mlops_roadmap.pdf', 'file_type': 'pdf'}, page_content='MLOps Roadmap\nYour Beginner-Friendly Guide to Machine Learning Operations\nby TechWorld with Nana'),
 Document(metadata={'producer': 'GPL Ghostscript 9.56.1', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': "D:20251016072905Z00'00'", 'moddate': "D:20251016072905Z00'00'", 'source': '../data/pdf/mlops_roadmap.pdf', 'total_pages': 21, 'page': 1, 'page_label': '2', 'source_file': 'mlops_roadmap.pdf', 'file_type': 'pdf'}, page_content="Provided by TechWorld with Nana\nI'm Nana, Co-Founder of TechWorld with Nana.\nAs a Cloud and DevOps engineer, I'm dedicated to \nhelping engineers build the most valuable and 

In [8]:
### Embedding and vector Stor DB

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embeddings using Sentence Transformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            Hugging Face model name for embedding manager
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence Transformer model"""
        try:
            print(f"Loading Sentence Transformer model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Get embedding dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Sentence Transformer model: {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for given texts

        Args:
            texts: list of texts to embed

        Returns:
            numpy array with embeddings with shape (len(texts), EMBEDDING_DIMENSIONS)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

### Initialize embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading Sentence Transformer model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2135.71it/s]


Model loaded successfully. Get embedding dimensions: 384


In [10]:
### Vector Store
class VectorStore:
    """Manage document embeddings in a chromaDB Vector Store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: name of the chromaDB collection
            persist_directory: directory to persist the vector_store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromaDB client and collection"""
        try:
            #Create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"Description": "PDF embeddings for RAG"}
            )
            print(f"Vector Store Initialized successfully. Collection: {self.collection_name}")
            print(f"Existing document in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray ):
        """
        Add documents and their embeddings to the vector store

        Args:
        documents: List of langchain documents
        embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} to the vector store....")

        #Preparing data for chromaDB

        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #Documents content
            document_text.append(doc.page_content)

            #Embeddings
            embeddings_list.append(embedding.tolist())

            #Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = document_text,
            )
            print(f"Added {len(documents)} documents to the vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore


Vector Store Initialized successfully. Collection: pdf_documents
Existing document in collection: 108


In [11]:
#Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
texts

['MLOps Roadmap\nYour Beginner-Friendly Guide to Machine Learning Operations\nby TechWorld with Nana',
 "Provided by TechWorld with Nana\nI'm Nana, Co-Founder of TechWorld with Nana.\nAs a Cloud and DevOps engineer, I'm dedicated to \nhelping engineers build the most valuable and highly-\ndemanded DevOps and Cloud skills.\nThrough my YouTube channel and my comprehensive \nDevOps bootcamps, I've helped 1,000,000s of \nengineers master the tools and concepts that drive \nmodern software development.\nAbout This MLOps Roadmap\nThis detailed guide provides a complete step-by-step roadmap to \nunderstanding and learning MLOps.\nWhether you're just starting out or looking to level up your existing skills, \nthis document gives you a structured guide to master MLOps.\nHappy learning!\nNana Janashia\nCo-Founder TechWorld with Nana",
 'Table of Contents\n1 What is MLOps and Why Do We Need It?\n2 The Real-World Problem MLOps Solves\n3 What Does an MLOps Workflow Look Like?\n4 Your MLOps Learning

In [12]:
#Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#Store in the vector DB
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 36 texts


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.53it/s]

Generated embeddings with shape: (36, 384)
Adding 36 to the vector store....
Added 36 documents to the vector store
Total documents in collection: 144


In [13]:
# Retriever pipeline from vector Store
class RAGRetriever:
    """Handles Query based retrieval from the vector store"""
    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vectorstore: vector store containing the document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents from the vector store

        Args:
            query: query to retrieve documents from
            top_k: number of top results to return
            score_threshold: Minimum similarity score to filter results
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Top results: {top_k}, Threshold: {score_threshold}")

        #Generate query embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #Search in vector store
        try:
            results = self.vectorstore.collection.query(
                query_embeddings = [query_embeddings.tolist()],
                n_results = top_k,
            )

            #Processed results
            retrieved_documents = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, documents, metadatas, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #Convert distance to similarity score, chromaDB uses cosine similarity
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_documents.append({
                            'id': doc_id,
                            'content': documents,
                            'metadata': metadatas,
                            'distance': distance,
                            'similarity_score': similarity_score,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_documents)} documents (after filtering): ")
            else:
                print("No documents retrieved")

            return retrieved_documents

        except Exception as e:
            print(f"Error retrieving documents: {e}" )
            return[]

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [14]:
rag_retriever.retrieve("What is MLOps")

Retrieving documents for query: What is MLOps
Top results: 5, Threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 106.64it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering): 


[{'id': 'doc_8741de5c_3',
  'content': "What is MLOps and Why We Need It?\nThe Kitchen Analogy:  \nThink of MLOps like a professional kitchen. Even the best chef (data scientist) can't serve food to hundreds \nof customers (users) without a proper kitchen setup. You need organized stations, quality equipment, food \nsafety standards, and a system to handle orders smoothly.\nThat's exactly what MLOps does for machine learning!\nMLOps stands for Machine Learning Operations. It's the practice of getting machine learning \nmodels from a data scientist's laptop into real products where actual users can benefit from \nthem - and keeping those models running smoothly over time.\nWhy Traditional Software Development Isn't Enough\nHere's the thing: building machine learning systems is different from building regular software.\nTraditional software is like building a house with a detailed blueprint. Once built, it works as designed \nunless you change the code.",
  'metadata': {'source_file': 'm

In [15]:
# Integration VectorDB context pipeline with LLM output
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the LLM via OpenRouter
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

llm = ChatOpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
    model="google/gemma-4-31b-it:free",
    temperature=0.1,
    max_tokens=1024,
)

def rag_simple(query, retriever, llm, top_k=3):
    # retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc['content'] for doc in results) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    # Generate the answer using LLM
    prompt = f"""Use the following context to answer the question concisely
        Context: {context}
        Question: {query}
        Answer:"""
    response = llm.invoke(prompt)
    return response.content

In [16]:
answer = rag_simple("What is MLOps?", rag_retriever, llm)
print(answer)

Retrieving documents for query: What is MLOps?
Top results: 3, Threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.30it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering): 


OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False, 'provider_error_code': '429', 'limit_source': 'upstream_provider_shared_pool', 'remedy_hint': 'Retry shortly, add your own provider key (https://openrouter.ai/settings/integrations), or route to another provider with provider routing: https://openrouter.ai/docs/features/provider-routing'}}, 'user_id': 'user_3ImHessbzCRL1sWGrBBoN2AsNu3'}

In [17]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: Hard Negative Mining Technqiues
Top results: 3, Threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering): 
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [19]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is MLOps?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: What is MLOps?
Top results: 3, Threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 118.32it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering): 
Streaming answer:
Use the following context to answer the question concisely.
Context:
What is MLOps and Why We Need It?
The Kitchen Analogy:  
Think of MLOps like a professional kitchen. Even the best chef (data scientist) can't serve food to hundreds 
of customers (users) without a proper kitchen setup. You need organized stations, qu

ality equipment, food 
safety standards, and a system to handle orders smoothly.
That's exactly what MLOps does for machine learning!
MLOps stands for Machine Learning Operations. It's the practice of getting machine learning 
models from a data scientist's laptop into real products where actual users can benefit from 
them - and keeping those models running smoothly over time.
Why Traditional Software Development Isn't Enough
Here's the thing: building machine learning systems is different from building regular software.
Traditional software is like building a house with a detailed blueprint. Once built, it works as designed 
unless you change the code.

What is MLOps and Why We Need It?
The Kitchen Analogy:  
Think of MLOps like a professional kitchen. Even the best chef (data scientist) can't serve food to hundreds 
of customers (users) without a proper kitchen setup. You need organized stations, quality equipment, food 
safety standards, and a system to handle orders smoothly.
That

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False, 'provider_error_code': '429', 'limit_source': 'upstream_provider_shared_pool', 'remedy_hint': 'Retry shortly, add your own provider key (https://openrouter.ai/settings/integrations), or route to another provider with provider routing: https://openrouter.ai/docs/features/provider-routing'}}, 'user_id': 'user_3ImHessbzCRL1sWGrBBoN2AsNu3'}